# Part 3 — The model and the quantization mathematics

The central object is a transformer whose weights are constrained to two values,
trained so that the checkpoint coming out of training *is* the hardware model. This
part defines it: the architecture, the binarization exactly as implemented (which is
not the textbook version), how a gradient is pushed through a sign function, why a
custom quantizer had to exist, how activations are quantized, and the cost model that
stands in for synthesis while experiments are still cheap.

It assumes the physics and not the quantization. Run from the repository root.

## The architecture

Each jet enters as N tokens of F features. A linear projection lifts them to an
embedding of dimension $d$, a learned positional table is added, and the tokens pass
through a stack of standard blocks — multi-head self-attention, then a two-layer
feed-forward network, each wrapped in a residual — before a global average pool over
constituents and a small head produce five logits.

Attention is composed explicitly from einsum primitives: per-head projections
$W_q, W_k, W_v$, the score contraction $QK^\top$ scaled by $1/\sqrt{d_\text{head}}$
(the scale folded into the softmax's exponential table, where it is free in hardware),
and a context contraction with $V$ followed by the output projection $W_o$
(`build_qat_model` in `bnjettag/code/hgq2/bnhgq2/qat.py`).

In [ ]:
import json

cfg = json.load(open("code/configs/r8-small-w1a8-stdnn.json"))
print(cfg["name"], "| era", cfg["era"])
for k, v in cfg["arch"].items():
    print(f"  {k:15s} {v}")

That is the round-8 flagship most of the current record is built on: $d = 32$, 2
blocks, 4 heads, FFN width 64, five classes, and **no normalization layers anywhere in
the graph** (`norm: "none"`). It is 19,201 parameters (`RESEARCH.md` §2, round-8
entry). A much larger 6,375,173-parameter configuration exists in the record as the
capacity reference (`bnjettag/results/ebops.md` §1).

### Why norm-free

The normalization kernel was the model's dominant consumer of both DSPs and LUTs in
every synthesis that carried one, and removing it turned out to *help* accuracy at this
scale: the configuration that is cheapest in hardware is also the best binary arm
(`RESEARCH.md` §2). Offline input standardization (part 2) does the conditioning work a
LayerNorm would otherwise do.

## Binary weights: absmean, not textbook sign

The textbook binarization is $q = \operatorname{sign}(W)$. What is implemented is the
BitNet-lineage *absmean* quantizer, applied per weight tensor with scalar statistics
(`bitnet_binary_ste` in `bnjettag/code/hgq2/bnhgq2/qat.py`; the identical numpy form in
`binarize.py`):

$$
\alpha = \operatorname{mean}(W), \qquad
W_c = W - \alpha, \qquad
\beta = \operatorname{mean}\lvert W_c \rvert + 10^{-6},
$$

$$
q_i = \begin{cases} +1 & W_{c,i} \ge 0 \\ -1 & W_{c,i} < 0 \end{cases}
\qquad
W_{\text{eff}} = \beta\, q \in \{-\beta, +\beta\}.
$$

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
W = rng.normal(size=64).astype(np.float32)     # latent full-precision weights

alpha = W.mean()                               # tensor DC offset
Wc    = W - alpha                              # centred
beta  = np.abs(Wc).mean() + 1e-6               # scalar scale, floored away from zero
q     = np.where(Wc >= 0.0, 1.0, -1.0)         # strict bipolar: ties go to +1, never 0
W_eff = beta * q

assert (q == 0).sum() == 0, "a zero state would make this ternary"
print(f"alpha = {alpha:+.6f}    beta = {beta:.6f}")
print("effective weight values:", np.unique(W_eff))
print("counts at -beta / +beta:", (q < 0).sum(), "/", (q > 0).sum())

Each piece earns its place.

- **Centering by $\alpha$** removes any DC offset before the sign is taken, so the
  $\pm 1$ split is balanced rather than biased toward whichever sign the
  initialization happened to favor.
- **The scale $\beta$** makes $\beta q$ approximate $W_c$ in an average sense; the
  $10^{-6}$ floor keeps it nonzero. It is the only full-precision remnant of the weight
  tensor, and it does not survive into the datapath: at export it is folded into the
  hardware graph layer class by layer class, so the multiplier arrays see pure $\pm 1$.
  That fold is what makes the zero-DSP property structural rather than lucky.
- **Bipolar sign, never zero.** Mathematically $\operatorname{sign}(0) = 0$, and a
  weight grid containing zero is *ternary* $\{-1, 0, +1\}$ — a different thesis. The
  implemented rule sends $W_{c,i} \ge 0$ to $+1$, so the grid is exactly
  $\{-\beta, +\beta\}$.

### The binary claim is enforced, not assumed

`binarize.py` counts `total_sign_zeros` across every layer, and the count must be 0 or
the binary claim is void. The training stage asserts at build time that every binary
layer's effective weights are exactly two values, symmetric, with no zero.

In [ ]:
src = open("code/bnhgq2/qat.py").read()
print(src[src.index("def effective_weight_values"):].rstrip())

## Training through a sign function

$\operatorname{sign}$ has zero derivative almost everywhere, so nothing would train if
the gradient were taken literally. The standard answer is the **straight-through
estimator**: keep full-precision *latent* master weights, quantize them in the forward
pass, and pretend in the backward pass that the quantizer was locally the identity. The
implementation is the classic stop-gradient composition,

$$
w_s = \frac{W_c}{\operatorname{sg}(\beta)}, \qquad
w_q = w_s + \operatorname{sg}(q - w_s), \qquad
\text{output weight} = w_q \cdot \beta ,
$$

where $\operatorname{sg}(\cdot)$ is stop-gradient. Forward, $w_q$ evaluates to exactly
$q$; backward, gradients flow into the latent tensor, and the optimizer updates
full-precision weights that drift smoothly even though the forward pass only ever sees
two values per tensor.

In [ ]:
# numpy has no autodiff, so this checks the FORWARD half of the composition only:
# the stop-gradient terms are the identity in value.
ws = Wc / beta
wq = ws + (q - ws)
print("forward value identical to beta * q:", np.allclose(wq * beta, W_eff))

One detail is a deliberate hardening rather than a transcription: the normalization
uses $\operatorname{sg}(\beta)$, not $\beta$. With the stop-gradient, the Jacobian into
the latent weights is $(I - \tfrac{1}{n}) + q\,\partial\beta/\partial W$ — every term
bounded, no $1/\beta$ factor anywhere.

The earlier formulation routed the gradient through $W_c/\beta$ with $\beta$
differentiable, which amplifies the backward pass by $1/\beta$ exactly when a kernel's
centered weights collapse and $\beta$ approaches its $10^{-6}$ floor. That
amplification was the verified cause of the 4-bit-activation training NaN, root-caused
2026-07-07; the forensic account is in part 8 (what broke).

## Why a custom quantizer exists at all

The training stack (HGQ2) ships fixed-point quantizers, and the obvious move is to
configure one at a width of 1 bit. That produces the wrong object. A signed fixed-point
grid at 1 bit is $\{-1, 0, +1\}$ — three states, because fixed-point grids always
contain zero — and rounding to it sends every latent weight below the rounding
threshold to exactly 0. In probe T1 this collapsed **4096 of 4096** latent weights of a
test layer to zero: not a degraded binary model but a dead ternary one.

Bipolar sets are not fixed-point sets, and no parametrization of a fixed-point
quantizer can produce $\{-\beta, +\beta\}$. Hence the custom path: `BitQDense` and
`BitQEinsumDense` subclass the stock layers, binarize their latent kernels with the
absmean STE in the forward pass, and keep a nominal 1-bit quantizer configuration
attached purely so that the cost accounting and the hls4ml converter both report 1-bit
weights.

## Activation quantization

Weights are one axis; how coarse the *activations* can get is the other. Activations
are quantized to a static per-tensor signed fixed-point grid
$\text{fixed}\langle b,\, 1{+}i \rangle$: $b$ total bits of which one is sign, $i$
integer bits, $f = b - 1 - i$ fractional bits, step $2^{-f}$, range
$[-2^{i},\ 2^{i} - 2^{-f}]$. Out-of-range values saturate (SAT) rather than wrap, and
rounding is round-half-even (RND_CONV), chosen so the trained grid is exactly the grid
the exported hardware applies.

The variant naming encodes both axes: **w1a8** is binary weights with 8-bit
activations, and the activation ladder A8 → A6 → A4 walks $b$ down at fixed binary
weights.

In [ ]:
import numpy as np

# signed fixed<bits, 1+i>: step 2**-f, saturating, round-half-even
def static_grid(x, bits, i):
    f = bits - 1 - i
    step = 2.0 ** -f
    lo, hi = -(2.0 ** i), 2.0 ** i - step
    return np.clip(np.round(x / step) * step, lo, hi)

x = np.array([-9.0, -0.031, 0.4999, 1.5, 7.9])
for bits, i in ((8, 3), (4, 2)):
    f = bits - 1 - i
    lo, hi = -(2.0 ** i), 2.0 ** i - 2.0 ** -f
    print(f"a{bits}  i={i} f={f}  step={2.0**-f:<7g} range=[{lo}, {hi}]")
    print("     ", static_grid(x, bits, i))

### Calibration sets the integer split

$i$ is set by calibration, not by hand: a forward pass over a real batch tapped at
every quantizer site chooses, per tensor, the $i$ that minimizes mean-squared
quantization error, floored so that $f \ge 1$ always (`calibrate_activations` in
`qat.py`). The rejected alternative is max-plus-margin, which over-ranges low-bit grids
— at A4 it would leave $f = 0$, an integer-only grid.

### Frozen grids fail at four bits

What happens after calibration became a finding in its own right. Grids calibrated once
and frozen are harmless at 8 bits, degrade at 6, and are catastrophic at 4: a
16.8-point validation AUC collapse, because at 16 levels the grid cannot follow the
drift of the activation distributions during training.

The rescue is a trainable-scale quantizer: total width pinned by a constraint, the
integer split $i$ trained with the model, so the grid re-aligns as distributions move.
The ablation and the drift counts — 3 of 83 grids moved at A8 against 12 of 83 at A4 —
are in `RESEARCH.md` §5″.

Later rounds go one step further and make the *bitwidths themselves* learnable under
explicit cost pressure: trainable $i$ and $f$ with an L1-style regularizer, annealed by
an EBOPs term in the loss, with the learned widths frozen for export so the shipped
grid is still static. In every mode the same invariant holds — the quantizer trained is
the quantizer deployed.

## EBOPs: the cost model before synthesis

Synthesis takes hours to days; training decisions need a cost axis available in
seconds. **EBOPs** (effective bit-operations) charge every scalar multiply the product
of its operand bitwidths, so a weight-activation MAC costs about $b_w \cdot b_a$ and
binarizing weights divides the multiply cost by the activation width.

The exception is architectural: the attention score and context contractions multiply
activations by activations, cost $b_a^2$, and are untouched by weight binarization.
That is the same structural fact that reappears in hardware as the attention DSP floor.

On the era-2 large model (6,375,173 parameters), closed-form convention, the arithmetic
gives W1A8 = 530.4M EBOPs against W8A8 = 4.06G — a factor of 7.65 rather than a clean
8x, because 0.65% of the MACs are those activation-times-activation attention products
(`bnjettag/results/ebops.md` §1, §2).

There are two EBOPs conventions in the record and they are never mixed in one column.
The closed-form convention implements the original HGQ definition (multiply term only).
The HGQ2-native convention is measured rather than derived: a tracing forward pass over
calibration data recomputes each layer's cost from its quantizer bitwidths, including
accumulator and table terms. The same large model therefore carries two legitimate
numbers of different sizes — 530.4M closed-form, 662.2M ± 0.2M native
(`RESEARCH.md` §6″) — and every table names its convention.

![Per-layer EBOPs of the 19,201-parameter round-8 flagship, HGQ2-native convention (a different model and a different convention from the large-model figures above): the eight largest layers, split between binary weight matmuls, attention activation-times-activation, and softmax. The act-times-act share is the piece weight binarization cannot touch. Generated by `bnjettag/code/plots/make_lab_slides.py` from the round-8 convert store, 2026-07-28.](figures/ebops_breakdown.png)

## The training recipe, with reasons

| Choice | Why |
|---|---|
| Adam with $\beta_2 = 0.98$ | Quantization noise makes gradient variance spiky — a weight crossing its sign threshold flips an entire column of the effective matrix — and heavier second-moment smoothing damps exactly that |
| One epoch linear warmup, then polynomial decay | The FINAL large-model campaign trained at peak $5\times10^{-5}$ (`RESEARCH.md` §5″); later campaigns re-probe the peak per scale rather than inheriting it |
| Early stopping on validation macro-OvR AUC, patience 15 as run, best weights restored | The monitor is the training-time signal; the ROC-test AUC (n = 260,000) is computed afterwards and separately |
| Gradient clipping by value, `clipvalue = 1.0`, not by global norm | Global-norm clipping sums squared gradients, which overflows float32 on this stack and destroyed the A4 arm at step 1 |
| Batch 256, weight decay 0.01 | Applied, like every optimizer step, to the latent full-precision weights, since those are the only weights that move |

## What the constraint costs

Holding every arm of a campaign identical except the quantization is what licenses the
summary: binarizing all weights costs 0.6 ROC-test macro-OvR AUC points at 6.4M
parameters, 7.6 at 19k, and 9.4 at 5.3k (each scale at its own tuned recipe,
seed-averaged, era-2 data), while 8-bit weights stay within about 0.7 points of full
precision everywhere (`RESEARCH.md` §2, exchange-curve entry as corrected 2026-07-16).

The binary constraint taxes capacity, not a fixed number of AUC points. What that
implies for where a 1-bit model is worth building is argued where the results are.

![The bits-for-parameters exchange curve: ROC-test macro-OvR AUC cost of weight binarization at three model scales (5.3k, 19k, 6.4M parameters), each trained at its own tuned recipe; 8-bit weights track full precision at every scale. From the frozen report figures, 2026-07-16, plotted from the verified round-7/7b and FINAL campaign tables of `RESEARCH.md` §2.](figures/bits_for_params_curve.png)